# Cross-Validation & Hyperparameter Tuning

Reliable model evaluation and systematic parameter optimisation.

## Why Cross-Validation?

Train/test split gives optimistic estimates. Cross-validation gives reliable performance estimates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, 
    cross_val_score, cross_validate, GridSearchCV, RandomizedSearchCV
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import time

np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## The Problem with Simple Train/Test Split

Performance depends on random split. Different splits give different results.

In [ ]:
# Generate data
X, y = make_classification(n_samples=200, n_features=4, n_classes=2, 
                          n_redundant=0, random_state=42)

# Test multiple random splits
accuracies = []
n_splits = 20

for i in range(n_splits):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=i
    )
    
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

# Plot variability
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(accuracies, bins=10, alpha=0.7, edgecolor='black')
plt.axvline(np.mean(accuracies), color='red', linestyle='--', 
            label=f'Mean: {np.mean(accuracies):.3f}')
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.title('Accuracy Distribution Across Random Splits')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, n_splits+1), accuracies, 'bo-', alpha=0.7)
plt.axhline(np.mean(accuracies), color='red', linestyle='--', 
            label=f'Mean: {np.mean(accuracies):.3f}')
plt.xlabel('Split Number')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Split Number')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Accuracy range: {min(accuracies):.3f} - {max(accuracies):.3f}")
print(f"Standard deviation: {np.std(accuracies):.3f}")
print("\nProblem: Single split gives unreliable estimate!")

## K-Fold Cross-Validation

Split data into K folds. Train on K-1, test on 1. Repeat K times. Average results.

In [ ]:
# Manual K-fold implementation
def manual_kfold_cv(X, y, k=5, model_class=LogisticRegression, **model_kwargs):
    """Manual K-fold cross-validation"""
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = []
    fold = 1
    
    for train_idx, val_idx in kf.split(X, y):
        # Split data
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]
        
        # Train model
        model = model_class(**model_kwargs)
        model.fit(X_train_fold, y_train_fold)
        
        # Evaluate
        y_pred = model.predict(X_val_fold)
        score = accuracy_score(y_val_fold, y_pred)
        scores.append(score)
        
        print(f"Fold {fold}: Accuracy = {score:.3f}")
        fold += 1
    
    return scores

# Test different K values
k_values = [3, 5, 10]
plt.figure(figsize=(15, 5))

for i, k in enumerate(k_values):
    scores = manual_kfold_cv(X, y, k=k, random_state=42)
    
    plt.subplot(1, 3, i+1)
    plt.bar(range(1, k+1), scores, alpha=0.7, edgecolor='black')
    plt.axhline(np.mean(scores), color='red', linestyle='--', 
                label=f'Mean: {np.mean(scores):.3f}')
    plt.xlabel('Fold')
    plt.ylabel('Accuracy')
    plt.title(f'{k}-Fold Cross-Validation')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0.7, 1.0)

plt.tight_layout()
plt.show()

# Compare with sklearn
model = LogisticRegression(random_state=42)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"\nSklearn 5-fold CV: {cv_scores}")
print(f"Mean accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

print("\nK-Fold Guidelines:")
print("- K=5 or K=10: Good balance of bias/variance")
print("- Higher K: Less bias, more variance, slower")
print("- Lower K: More bias, less variance, faster")

## Stratified K-Fold

Maintains class distribution in each fold. Essential for imbalanced data.

In [ ]:
# Create imbalanced data
X_imb, y_imb = make_classification(n_samples=300, n_features=4, n_classes=2,
                                  weights=[0.8, 0.2], random_state=42)

print("Class distribution in full dataset:")
unique, counts = np.unique(y_imb, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count} samples ({count/len(y_imb)*100:.1f}%)")

# Regular K-fold vs Stratified K-fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nFold distributions:")
print("Fold | Regular K-Fold | Stratified K-Fold")
print("-----|---------------|-------------------")

for fold, ((train_reg, val_reg), (train_strat, val_strat)) in enumerate(zip(kf.split(X_imb, y_imb), skf.split(X_imb, y_imb))):
    # Regular K-fold
    y_val_reg = y_imb[val_reg]
    reg_dist = np.bincount(y_val_reg) / len(y_val_reg)
    
    # Stratified K-fold
    y_val_strat = y_imb[val_strat]
    strat_dist = np.bincount(y_val_strat) / len(y_val_strat)
    
    print(f"  {fold+1}  |    {reg_dist[1]:.2f}       |      {strat_dist[1]:.2f}")

# Performance comparison
model = LogisticRegression(random_state=42)

regular_scores = cross_val_score(model, X_imb, y_imb, cv=kf, scoring='accuracy')
stratified_scores = cross_val_score(model, X_imb, y_imb, cv=skf, scoring='accuracy')

print(f"\nRegular K-fold CV: {regular_scores.mean():.3f} ± {regular_scores.std():.3f}")
print(f"Stratified K-fold CV: {stratified_scores.mean():.3f} ± {stratified_scores.std():.3f}")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, 6), regular_scores, alpha=0.7, label='Regular K-fold', color='blue')
plt.axhline(regular_scores.mean(), color='blue', linestyle='--')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Regular K-Fold CV')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(range(1, 6), stratified_scores, alpha=0.7, label='Stratified K-fold', color='green')
plt.axhline(stratified_scores.mean(), color='green', linestyle='--')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Stratified K-Fold CV')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nStratified CV:")
print("- Maintains class proportions in each fold")
print("- Reduces variance in performance estimates")
print("- Essential for imbalanced datasets")

## Hyperparameter Tuning

Systematically find best model parameters using cross-validation.

In [ ]:
# Generate data for tuning
X_tune, y_tune = make_classification(n_samples=500, n_features=10, n_classes=2, 
                                    n_redundant=5, random_state=42)
X_train_tune, X_test_tune, y_train_tune, y_test_tune = train_test_split(
    X_tune, y_tune, test_size=0.2, random_state=42
)

# Manual grid search
def manual_grid_search(X, y, param_grid, cv=5):
    """Manual grid search with CV"""
    best_score = 0
    best_params = None
    results = []
    
    from itertools import product
    param_combinations = list(product(*param_grid.values()))
    
    for params in param_combinations:
        param_dict = dict(zip(param_grid.keys(), params))
        
        model = LogisticRegression(**param_dict, random_state=42, max_iter=1000)
        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
        mean_score = scores.mean()
        
        results.append({
            'params': param_dict,
            'mean_score': mean_score,
            'std_score': scores.std()
        })
        
        if mean_score > best_score:
            best_score = mean_score
            best_params = param_dict
    
    return best_params, best_score, results

# Define parameter grid
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # Required for l1 penalty
}

# Manual search
print("Running manual grid search...")
start_time = time.time()
best_params_manual, best_score_manual, results_manual = manual_grid_search(
    X_train_tune, y_train_tune, param_grid
)
manual_time = time.time() - start_time

print(f"Manual search time: {manual_time:.2f} seconds")
print(f"Best params: {best_params_manual}")
print(f"Best CV score: {best_score_manual:.3f}")

# Sklearn GridSearchCV
print("\nRunning sklearn GridSearchCV...")
start_time = time.time()
model = LogisticRegression(random_state=42, max_iter=1000)
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_tune, y_train_tune)
sklearn_time = time.time() - start_time

print(f"Sklearn search time: {sklearn_time:.2f} seconds")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")

# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test_tune)
test_accuracy = accuracy_score(y_test_tune, y_pred_test)

print(f"\nTest set accuracy: {test_accuracy:.3f}")

# Plot results
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
l1_scores = []
l2_scores = []

for c in C_values:
    # L1
    model_l1 = LogisticRegression(C=c, penalty='l1', solver='liblinear', random_state=42)
    scores_l1 = cross_val_score(model_l1, X_train_tune, y_train_tune, cv=5)
    l1_scores.append(scores_l1.mean())
    
    # L2
    model_l2 = LogisticRegression(C=c, penalty='l2', random_state=42)
    scores_l2 = cross_val_score(model_l2, X_train_tune, y_train_tune, cv=5)
    l2_scores.append(scores_l2.mean())

plt.figure(figsize=(10, 6))
plt.semilogx(C_values, l1_scores, 'b-o', label='L1 penalty', linewidth=2)
plt.semilogx(C_values, l2_scores, 'r-o', label='L2 penalty', linewidth=2)
plt.xlabel('C (Inverse Regularisation Strength)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Hyperparameter Tuning: C and Penalty Type')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nGrid Search:")
print("- Exhaustive search over parameter combinations")
print("- Reliable but slow for large grids")
print("- Use when you have few parameters to tune")

## Randomized Search

Sample random parameter combinations. Faster than grid search for large spaces.

In [ ]:
# Decision tree hyperparameter tuning
tree_param_dist = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2', None]
}

# Grid search (expensive)
print("Grid search combinations:", 
      len(tree_param_dist['max_depth']) * 
      len(tree_param_dist['min_samples_split']) * 
      len(tree_param_dist['min_samples_leaf']) * 
      len(tree_param_dist['criterion']) * 
      len(tree_param_dist['max_features']))

# Randomized search
print("\nRunning RandomizedSearchCV (50 iterations)...")
start_time = time.time()

tree_model = DecisionTreeClassifier(random_state=42)
random_search = RandomizedSearchCV(
    tree_model, tree_param_dist, n_iter=50, cv=5, 
    scoring='accuracy', random_state=42, n_jobs=-1
)
random_search.fit(X_train_tune, y_train_tune)

random_time = time.time() - start_time

print(f"Randomized search time: {random_time:.2f} seconds")
print(f"Best params: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.3f}")

# Evaluate on test set
best_tree = random_search.best_estimator_
y_pred_tree = best_tree.predict(X_test_tune)
tree_test_acc = accuracy_score(y_test_tune, y_pred_tree)

print(f"Test accuracy: {tree_test_acc:.3f}")

# Compare with default tree
default_tree = DecisionTreeClassifier(random_state=42)
default_scores = cross_val_score(default_tree, X_train_tune, y_train_tune, cv=5)

print(f"\nDefault tree CV score: {default_scores.mean():.3f} ± {default_scores.std():.3f}")
print(f"Tuned tree CV score: {random_search.best_score_:.3f}")
print(f"Improvement: {random_search.best_score_ - default_scores.mean():.3f}")

# Plot parameter importance
results_df = pd.DataFrame(random_search.cv_results_)
param_cols = [col for col in results_df.columns if col.startswith('param_')]

plt.figure(figsize=(15, 10))

# Plot scores vs parameters
for i, param in enumerate(['param_max_depth', 'param_min_samples_split', 'param_min_samples_leaf']):
    plt.subplot(2, 3, i+1)
    if param in results_df.columns:
        param_values = results_df[param]
        scores = results_df['mean_test_score']
        plt.scatter(param_values, scores, alpha=0.6)
        plt.xlabel(param.replace('param_', ''))
        plt.ylabel('CV Score')
        plt.title(f'Score vs {param.replace("param_", "")}')
        plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRandomized Search:")
print("- Samples random parameter combinations")
print("- Faster than grid search for large parameter spaces")
print("- Good when you have many parameters to tune")
print("- May miss optimal combination, but finds good ones")

## Validation Curves

Plot performance vs hyperparameter values to understand bias-variance tradeoff.

In [ ]:
from sklearn.model_selection import validation_curve
import pandas as pd

# Validation curve for decision tree depth
param_range = [1, 2, 3, 4, 5, 7, 10, 15, 20, None]

train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=42), X_train_tune, y_train_tune,
    param_name='max_depth', param_range=param_range,
    cv=5, scoring='accuracy', n_jobs=-1
)

# Calculate means and stds
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Plot validation curve
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(param_range[:-1], train_mean[:-1], 'b-o', label='Training score', linewidth=2)
plt.fill_between(param_range[:-1], train_mean[:-1] - train_std[:-1], 
                 train_mean[:-1] + train_std[:-1], alpha=0.1, color='blue')
plt.plot(param_range[:-1], val_mean[:-1], 'r-o', label='Validation score', linewidth=2)
plt.fill_between(param_range[:-1], val_mean[:-1] - val_std[:-1], 
                 val_mean[:-1] + val_std[:-1], alpha=0.1, color='red')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Validation Curve: Decision Tree Depth')
plt.legend()
plt.grid(True, alpha=0.3)

# Bias-variance analysis
plt.subplot(2, 2, 2)
bias = 1 - val_mean[:-1]  # Higher validation error = more bias
variance = train_std[:-1]  # Higher training std = more variance
plt.plot(param_range[:-1], bias, 'g-o', label='Bias (1 - val_score)', linewidth=2)
plt.plot(param_range[:-1], variance, 'm-o', label='Variance (train_std)', linewidth=2)
plt.xlabel('Max Depth')
plt.ylabel('Error/Std')
plt.title('Bias-Variance Analysis')
plt.legend()
plt.grid(True, alpha=0.3)

# Learning curves
from sklearn.model_selection import learning_curve

train_sizes, train_scores_lc, val_scores_lc = learning_curve(
    DecisionTreeClassifier(max_depth=5, random_state=42), 
    X_train_tune, y_train_tune, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean_lc = np.mean(train_scores_lc, axis=1)
train_std_lc = np.std(train_scores_lc, axis=1)
val_mean_lc = np.mean(val_scores_lc, axis=1)
val_std_lc = np.std(val_scores_lc, axis=1)

plt.subplot(2, 2, 3)
plt.plot(train_sizes, train_mean_lc, 'b-o', label='Training score', linewidth=2)
plt.fill_between(train_sizes, train_mean_lc - train_std_lc, 
                 train_mean_lc + train_std_lc, alpha=0.1, color='blue')
plt.plot(train_sizes, val_mean_lc, 'r-o', label='Validation score', linewidth=2)
plt.fill_between(train_sizes, val_mean_lc - val_std_lc, 
                 val_mean_lc + val_std_lc, alpha=0.1, color='red')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.title('Learning Curve')
plt.legend()
plt.grid(True, alpha=0.3)

# Overfitting detection
plt.subplot(2, 2, 4)
gap = train_mean_lc - val_mean_lc
plt.plot(train_sizes, gap, 'purple', linewidth=2)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Training Set Size')
plt.ylabel('Train-Val Gap')
plt.title('Overfitting Indicator (Train-Val Gap)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nValidation Curves:")
print("- Show bias-variance tradeoff")
print("- Help choose optimal hyperparameter values")
print("- Identify overfitting/underfitting regions")

print("\nLearning Curves:")
print("- Show performance vs training data size")
print("- Large gap = overfitting")
print("- Both curves low = underfitting")
print("- Both curves high = good fit")

## Key Takeaways

- **Cross-validation** gives reliable performance estimates. Use K=5 or K=10.
- **Stratified CV** maintains class balance. Essential for imbalanced data.
- **Grid search** exhaustive but slow. Good for small parameter spaces.
- **Random search** faster for large spaces. May miss optimal but finds good solutions.
- **Validation curves** show bias-variance tradeoff. Help choose hyperparameters.
- **Learning curves** diagnose overfitting/underfitting. Guide data collection.

## Exercises

1. **CV Variance**: Why does cross-validation reduce variance compared to single train/test split?

2. **Stratification**: When should you use StratifiedKFold instead of regular KFold?

3. **Grid vs Random**: For a model with 4 hyperparameters, each with 5 values, which search method would you choose?

4. **Validation Curves**: You see validation score peaks at hyperparameter value X, then decreases. What does this indicate?

5. **Learning Curves**: Training and validation curves are both low and close together. What should you do?

## Solutions

1. **CV Variance**: CV averages over multiple train/test splits, reducing impact of any single bad split.

2. **Stratification**: Always for classification with imbalanced classes. Maintains class proportions in each fold.

3